<a href="https://colab.research.google.com/github/Foxokiso/hermes-agent/blob/main/COMFYUI_GAY621_A100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ComfyUI + gay621 (maleE621) — A100

Opens **from GitHub**. Checkpoint stays on Drive. Do not re-upload the 6.46 GB file.

| | |
|---|---|
| GPU | Runtime → **A100** + High-RAM (T4 can run SDXL slowly; A100 is the path) |
| Code | `comfyanonymous/ComfyUI` cloned here. **No ComfyUI-Manager** (REST `/prompt` + Manager tqdm = Errno 22) |
| Weights | Drive `MyDrive/models/gay621FurryMaleFocus_gay621XLV10.safetensors` (id `10CqDOkxJAfzevb3gtqkADMhprJS5nwhB`, 6938040736 bytes) |
| IN | Drive `MyDrive/AI-inputs/` (images/masks only) |
| JOBS | Drive `MyDrive/AI-jobs/*.json` — Comfy **API** format (`class_type` keys) |
| OUT | Drive `MyDrive/AI-outputs/` |

This is **not** E621 Bridge v2 (Flask `/generate`). Same checkpoint, Comfy graph jobs.

**Run:** Connect A100 → Run all → authorize Drive. First session copies the checkpoint (minutes). Later jobs: drop JSON in `AI-jobs/`, re-run **the last cell only**. Do not re-run the server cell.

UI tunnel is optional. Headless jobs hit `http://127.0.0.1:8188`.


In [ ]:
# 0) GPU gate
import torch, sys
print("python", sys.version.split()[0])
assert torch.cuda.is_available(), "No CUDA. Runtime → Change runtime type → A100 GPU."
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / (1024**3)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name}  VRAM: {vram:.1f} GB")
if vram < 15:
    raise SystemExit(
        f"Need a real GPU. This box is {vram:.1f}GB ({name}). "
        "Runtime → Change runtime type → A100."
    )
if vram < 24:
    print("WARN: under 24GB. SDXL will run; A100 is the intended runtime.")
else:
    print("A100-class OK.")


In [ ]:
# 1) Drive = files. Code stays on GitHub.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
for p in (DRIVE / "models", DRIVE / "AI-inputs", DRIVE / "AI-jobs", DRIVE / "AI-outputs"):
    p.mkdir(parents=True, exist_ok=True)
    print(p, "ok")


In [ ]:
# 2) Clone ComfyUI from GitHub. No Manager.
import os
os.chdir("/content")
if not os.path.isdir("/content/ComfyUI/.git"):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
else:
    print("ComfyUI already cloned")
%cd /content/ComfyUI
!git log -1 --oneline
!pip install -q -r requirements.txt
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb > /dev/null 2>&1 || (apt-get install -f -y -qq && dpkg -i /tmp/cloudflared.deb)
print("ComfyUI + cloudflared ready. Manager not installed on purpose.")


In [ ]:
# 3) Pin gay621. Size-check. Do not glob. Do not re-download.
import os, shutil
from pathlib import Path

NEED = 6938040736
CKPT = "gay621FurryMaleFocus_gay621XLV10.safetensors"
SRC = Path("/content/drive/MyDrive/models") / CKPT
DST = Path("/content/ComfyUI/models/checkpoints") / CKPT
INP = Path("/content/ComfyUI/input")
DST.parent.mkdir(parents=True, exist_ok=True)
INP.mkdir(parents=True, exist_ok=True)

if not SRC.is_file():
    raise FileNotFoundError(
        "Missing %s. Stop. Do not download a substitute." % SRC
    )
src_sz = SRC.stat().st_size
print("Drive ckpt bytes:", src_sz)
if src_sz != NEED:
    print("WARN: Drive size is not the known 6938040736. Copying anyway.")

if DST.is_file() and DST.stat().st_size == src_sz:
    print("Already in place:", DST, DST.stat().st_size)
else:
    if DST.exists():
        DST.unlink()
    gb = src_sz / (1024 ** 3)
    print("Copying", SRC, "(%.2f GB). Minutes. Do not stop the cell." % gb)
    shutil.copy2(SRC, DST)
    print("Copied", DST.stat().st_size)

n = 0
for f in Path("/content/drive/MyDrive/AI-inputs").glob("*"):
    if f.is_file() and f.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"):
        shutil.copy2(f, INP / f.name)
        n += 1
print("Synced %d image(s) into ComfyUI/input (png/jpg/webp only)" % n)


In [ ]:
# 4) Start ComfyUI once. Wait on /system_stats. Optional UI tunnel.
import os, subprocess, threading, time, re, urllib.request, json
from datetime import datetime, timezone
from pathlib import Path

os.environ["TQDM_DISABLE"] = "1"
URL = "http://127.0.0.1:8188"
LOG = Path("/content/ComfyUI/cloudflared.log")

def up():
    try:
        urllib.request.urlopen(URL + "/system_stats", timeout=2)
        return True
    except Exception:
        return False

if up():
    print("ComfyUI already on 8188 — not launching a second main.py")
else:
    env = os.environ.copy()
    env["TQDM_DISABLE"] = "1"
    proc = subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd="/content/ComfyUI",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    def stream():
        for line in proc.stdout:
            print(line, end="")
    threading.Thread(target=stream, daemon=True).start()

for i in range(90):
    if up():
        print("ComfyUI /system_stats OK")
        break
    time.sleep(2)
else:
    raise SystemExit("ComfyUI did not start. Do not re-run this cell blindly — check the log above.")

# Optional UI. Jobs do not need this.
if not LOG.exists() or LOG.stat().st_size == 0:
    LOG.write_text("")
    subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
        cwd="/content/ComfyUI",
        stdout=open(LOG, "ab"),
        stderr=subprocess.STDOUT,
    )
url = None
for _ in range(20):
    time.sleep(1)
    if LOG.exists():
        for line in LOG.read_text(errors="ignore").splitlines():
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m:
                url = m.group(0)
                break
    if url:
        break
print("=" * 60)
if url:
    print("ComfyUI UI (optional):", url)
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    Path("/content/drive/MyDrive/comfyui_tunnel_url.txt").write_text(f"{stamp}\n{url}\n")
    print("Wrote MyDrive/comfyui_tunnel_url.txt (does not clobber Flask tunnel_url.txt)")
else:
    print("No UI URL yet. Headless jobs still work on 127.0.0.1:8188. Do not re-run this cell.")
print("=" * 60)


In [ ]:
# 5) HEADLESS JOBS — re-run THIS cell after dropping JSON in AI-jobs/
import json, glob, os, time, shutil, uuid, urllib.request
from pathlib import Path

URL = "http://127.0.0.1:8188"
JOBS = Path("/content/drive/MyDrive/AI-jobs")
OUT = Path("/content/drive/MyDrive/AI-outputs")
DONE = JOBS / "_done"
FAIL = JOBS / "_failed"
OUT.mkdir(parents=True, exist_ok=True)
DONE.mkdir(parents=True, exist_ok=True)
FAIL.mkdir(parents=True, exist_ok=True)

def api(path, data=None, timeout=30):
    if data is None:
        with urllib.request.urlopen(URL + path, timeout=timeout) as r:
            return json.loads(r.read())
    req = urllib.request.Request(
        URL + path,
        data=json.dumps(data).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

for _ in range(90):
    try:
        api("/system_stats")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI not up — run the server cell once, then this cell.")

jobs = sorted(p for p in JOBS.glob("*.json") if p.is_file())
print(f"{len(jobs)} job(s) in AI-jobs/ (not _done/_failed)")

for job in jobs:
    name = job.stem
    try:
        wf = json.loads(job.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"[{name}] bad JSON: {e}")
        shutil.move(str(job), str(FAIL / job.name))
        continue
    if isinstance(wf, dict) and "nodes" in wf and "links" in wf:
        print(f"[{name}] editor format — export API format. skipped")
        shutil.move(str(job), str(FAIL / job.name))
        continue
    if not isinstance(wf, dict) or not any(isinstance(v, dict) and "class_type" in v for v in wf.values()):
        print(f"[{name}] not Comfy API format. skipped")
        shutil.move(str(job), str(FAIL / job.name))
        continue
    try:
        resp = api("/prompt", {"prompt": wf, "client_id": str(uuid.uuid4())})
    except Exception as e:
        print(f"[{name}] POST /prompt failed: {e}")
        shutil.move(str(job), str(FAIL / job.name))
        continue
    if resp.get("error") or resp.get("node_errors"):
        print(f"[{name}] rejected:", json.dumps({k: resp.get(k) for k in ("error", "node_errors")})[:800])
        shutil.move(str(job), str(FAIL / job.name))
        continue
    pid = resp.get("prompt_id")
    print(f"[{name}] queued {pid}")
    saved = 0
    for _ in range(600):
        time.sleep(2)
        h = api("/history/" + pid)
        if pid not in h:
            continue
        e = h[pid]
        st = (e.get("status") or {}).get("status_str")
        if st == "error":
            print(f"[{name}] ERROR:", json.dumps(e.get("status"), default=str)[:800])
            shutil.move(str(job), str(FAIL / job.name))
            break
        for node in (e.get("outputs") or {}).values():
            for img in node.get("images") or []:
                src = Path("/content/ComfyUI/output") / (img.get("subfolder") or "") / img["filename"]
                dst = OUT / f"{name}_{img['filename']}"
                if src.is_file():
                    shutil.copy2(src, dst)
                    saved += 1
                    print(f"[{name}] saved → AI-outputs/{dst.name}")
        else:
            if saved or st == "success" or (e.get("outputs") is not None):
                shutil.move(str(job), str(DONE / job.name))
                if saved == 0:
                    print(f"[{name}] finished with 0 images")
            break
    else:
        print(f"[{name}] timed out")
        shutil.move(str(job), str(FAIL / job.name))

print("Jobs complete. Check MyDrive/AI-outputs/")
